In [ ]:
# One-cell: 4 PDFs (RP vs dim/var, All tasks pooled vs dim/var) with simpler titles and lighter n-labels

import os, re, json
from pathlib import Path
import numpy as np, pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from matplotlib.ticker import MultipleLocator, FuncFormatter

BASE = Path('/scratch/gpfs/nb0564/vlm_reasoning')
OUT = BASE / 'plots' / 'publication'
OUT.mkdir(parents=True, exist_ok=True)

DOMAINS = [
  ('regular_polygons', BASE/'data'/'regular_polygons_corrected', BASE/'output'/'None'/'gemini-flash_corrected', r'^regular_polygons_(\d)dim_var(\d+)_dim\d+\.csv$'),
  ('glyphs',           BASE/'data'/'glyphs',                   BASE/'output'/'glyphs'/'gemini-flash',           r'^glyphs_dim(\d)_var(\d+)_dim\d+\.csv$'),
  ('nuts_and_bolts',   BASE/'data'/'nuts_and_bolts',           BASE/'output'/'nuts_and_bolts'/'gemini-flash',   r'^nab_dim(\d)_var(\d+)_dim\d+\.csv$'),
  ('totems',           BASE/'data'/'totems',                   BASE/'output'/'totems'/'gemini-flash',           r'^totems_(\d)mod_var(\d+)_dim\d+\.csv$'),
]

# Clean, compact styling
sns.set_theme(style='white')
plt.rcParams.update({
  'figure.figsize': (3.5, 3.1),
  'font.family': 'DejaVu Sans',
  'font.size': 9.5,
  'axes.labelsize': 9.5,
  'axes.titlesize': 9.5,
  'xtick.labelsize': 9,
  'ytick.labelsize': 9,
  'axes.linewidth': 0.8,
  'lines.linewidth': 1.7,
  'savefig.bbox': 'tight',
  'savefig.pad_inches': 0.03,
})
COL_RP   = '#1f78b4'
COL_POOL = '#6F4C9B'

def read_trials_map(path: Path):
  if not path or not path.exists(): return {}
  idx2odd = {}
  with open(path) as f:
    for ln in f:
      try:
        j = json.loads(ln)
        idx2odd[int(j['trial_idx'])] = int(j['oddball_idx'])
      except Exception:
        pass
  return idx2odd

# Build per-trial accuracy
records = []
for dom, ddir, odir, pattern in DOMAINS:
  pat = re.compile(pattern)
  for f in sorted(odir.glob('*.csv')):
    m = pat.search(f.name)
    if not m: continue
    lvl, var = int(m.group(1)), int(m.group(2))

    trials_path = None
    for cand in [ddir/f'dim{lvl}'/f'var{var}'/'trials.jsonl', ddir/f'{lvl}mod'/f'var{var}'/'trials.jsonl']:
      if cand.exists(): trials_path = cand; break

    try: df = pd.read_csv(f)
    except Exception: continue
    if 'answer' not in df.columns: continue
    if 'trial_idx' not in df.columns:
      df = df.reset_index().rename(columns={'index': 'trial_idx'})

    if 'oddball_idx' in df.columns:
      corr = (df['answer'].astype(str) == df['oddball_idx'].astype(str)).astype(int)
    else:
      idx2odd = read_trials_map(trials_path)
      df['__odd'] = df['trial_idx'].map(lambda t: idx2odd.get(int(t)))
      corr = (df['answer'].astype(str) == df['__odd'].astype(str)).astype(int)

    records.append(pd.DataFrame({
      'domain': dom, 'dim': lvl, 'var': var, 'trial_idx': df['trial_idx'], 'is_correct': corr,
    }))

acc = pd.concat(records, ignore_index=True) if records else pd.DataFrame(columns=['domain','dim','var','trial_idx','is_correct'])

def ci_boot(values, B=2500, alpha=0.05):
  a = np.asarray(values, float)
  if a.size == 0: return (np.nan, np.nan, np.nan)
  boots = [np.mean(np.random.choice(a, size=a.size, replace=True)) for _ in range(B)]
  return float(a.mean()), float(np.percentile(boots, 100*alpha/2)), float(np.percentile(boots, 100*(1-alpha/2)))

def style_axes(ax, title=None):
  if title: ax.set_title(title, pad=6)
  ax.spines['top'].set_visible(False)
  ax.spines['right'].set_visible(False)
  ax.yaxis.set_major_locator(MultipleLocator(0.1))
  ax.yaxis.set_minor_locator(MultipleLocator(0.05))
  ax.yaxis.set_major_formatter(FuncFormatter(lambda y, _: f'{int(round(y*100))}%'))
  ax.grid(axis='y', which='major', color='#b0b0b0', alpha=0.24, linewidth=0.6)
  ax.grid(axis='y', which='minor', alpha=0.14, linewidth=0.4)
  ax.set_ylim(0, 1)
  ax.set_xlabel(ax.get_xlabel(), labelpad=5)
  ax.set_ylabel('Accuracy', labelpad=5)
  ax.margins(x=0.04)

def draw_curve(df, xcol, color, title, outname):
  pts = []
  for x, g in sorted(df.groupby(xcol)):
    m, lo, hi = ci_boot(g['is_correct'].values)
    pts.append((x, m, lo, hi, len(g)))
  P = pd.DataFrame(pts, columns=[xcol,'mean','lo','hi','n']).sort_values(xcol)

  fig, ax = plt.subplots()
  if not P.empty:
    ax.plot(P[xcol], P['mean'], color=color, marker='o', markersize=4.2, lw=1.7)
    ax.fill_between(P[xcol], P['lo'], P['hi'], color=color, alpha=0.15, linewidth=0)
    # smaller, lighter n-labels slightly below points
    for _, r in P.iterrows():
      ax.annotate(f"n={int(r['n'])}", (r[xcol], r['mean']),
                  xytext=(0, -10), textcoords='offset points',
                  ha='center', va='top', fontsize=7, color='#777777')
  ax.set_xlabel(xcol.capitalize())
  style_axes(ax, title)
  plt.savefig(OUT / outname)
  plt.close(fig)
  print('Saved:', OUT / outname)

# Regular polygons
rp = acc[acc['domain'] == 'regular_polygons']
draw_curve(rp, 'dim', COL_RP,   'Accuracy vs dimension (regular polygons)', 'regular_polygons_accuracy_vs_dim.pdf')
draw_curve(rp, 'var', COL_RP,   'Accuracy vs variance (regular polygons)',  'regular_polygons_accuracy_vs_var.pdf')

# All tasks pooled
draw_curve(acc, 'dim', COL_POOL, 'Accuracy vs dimension (all tasks)', 'all_tasks_accuracy_vs_dim.pdf')
draw_curve(acc, 'var', COL_POOL, 'Accuracy vs variance (all tasks)',  'all_tasks_accuracy_vs_var.pdf')

Saved: /scratch/gpfs/nb0564/vlm_reasoning/plots/publication/regular_polygons_accuracy_vs_dim.pdf
Saved: /scratch/gpfs/nb0564/vlm_reasoning/plots/publication/regular_polygons_accuracy_vs_var.pdf
Saved: /scratch/gpfs/nb0564/vlm_reasoning/plots/publication/all_tasks_accuracy_vs_dim.pdf
Saved: /scratch/gpfs/nb0564/vlm_reasoning/plots/publication/all_tasks_accuracy_vs_var.pdf
